In [1]:
!pip install transformers datasets accelerate evaluate peft


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [16]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

In [14]:
pip install -U fsspec==2023.9.2


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from datasets import load_dataset

pubmedqa = load_dataset("pubmed_qa", "pqa_labeled", split="train[:10%]") 
pubmedqa = pubmedqa.filter(lambda x: x["long_answer"] is not None)

pubmedqa = pubmedqa.map(lambda x: {
    "input_text": f"question: {x['question']} context: {x['context']}",
    "target_text": x["long_answer"]
})

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_dataset

medmcqa = load_dataset("medmcqa", split="train[:10%]")

medmcqa = medmcqa.filter(lambda x: x["question"] and x["opa"] and x["opb"] and x["opc"] and x["opd"] and x["cop"] is not None)

def format_mcq(example):
    input_text = (
        f"question: {example['question']} "
        f"options: A. {example['opa']} B. {example['opb']} C. {example['opc']} D. {example['opd']}"
    )
    options = [example["opa"], example["opb"], example["opc"], example["opd"]]

    try:
        target_text = options[int(example["cop"])]
    except (ValueError, IndexError):
        target_text = "Unknown"

    return {
        "input_text": input_text,
        "target_text": target_text
    }

medmcqa = medmcqa.map(format_mcq)


Filter:   0%|          | 0/18282 [00:00<?, ? examples/s]

Map:   0%|          | 0/18282 [00:00<?, ? examples/s]

In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

def tokenize(example):
    model_input = tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )
    labels = tokenizer(
        example["target_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    model_input["labels"] = labels["input_ids"]
    return model_input

tokenized_pubmedqa = pubmedqa.map(tokenize, remove_columns=pubmedqa.column_names)
tokenized_medmcqa = medmcqa.map(tokenize, remove_columns=medmcqa.column_names)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/18282 [00:00<?, ? examples/s]

In [18]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

def train_on_dataset(tokenized_dataset, model_save_path, model, tokenizer):
    training_args = Seq2SeqTrainingArguments(
        output_dir=model_save_path,
        per_device_train_batch_size=4,
        learning_rate=3e-4,
        num_train_epochs=2,
        weight_decay=0.01,
        predict_with_generate=True,
        logging_dir="./logs",
        logging_steps=10,
        save_total_limit=1,
        save_strategy="epoch"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer
    )

    trainer.train()
    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)


In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer

model_pubmedqa = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
tokenizer_pubmedqa = AutoTokenizer.from_pretrained("google/flan-t5-base")

train_on_dataset(
    tokenized_dataset=tokenized_pubmedqa,
    model_save_path="./flan-t5-pubmedqa1",
    model=model_pubmedqa,
    tokenizer=tokenizer_pubmedqa
)

/tmp/ipython-input-18-2597966932.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
10,11.448700
20,3.538600
30,2.427900
40,1.744400
50,1.543300


In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer


model_medmcqa = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
tokenizer_medmcqa = AutoTokenizer.from_pretrained("google/flan-t5-base")


train_on_dataset(
    tokenized_dataset=tokenized_medmcqa,
    model_save_path="./flan-t5-medmcqa1",
    model=model_medmcqa,
    tokenizer=tokenizer_medmcqa
)

/tmp/ipython-input-18-2597966932.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
10,21.987800
20,19.947300
30,4.497600
40,1.457800
50,0.258500
60,0.078600
70,0.034600
80,0.038700
90,0.031300
100,0.025700


In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch


model_pubmedqa = T5ForConditionalGeneration.from_pretrained("./flan-t5-pubmedqa1")
tokenizer_pubmedqa = AutoTokenizer.from_pretrained("./flan-t5-pubmedqa1")

model_medmcqa = T5ForConditionalGeneration.from_pretrained("./flan-t5-medmcqa1")
tokenizer_medmcqa = AutoTokenizer.from_pretrained("./flan-t5-medmcqa1")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_pubmedqa.to(device)
model_medmcqa.to(device)


/Users/pavanreddy/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [2]:
from datasets import load_dataset

pubmedqa = load_dataset("pubmed_qa", "pqa_labeled", split="train[:100]")
medmcqa = load_dataset("medmcqa", split="train[:100]")

pubmedqa_test = []
for item in pubmedqa:
    question = item["question"]
    context = item["context"]["contexts"][0] if item["context"]["contexts"] else ""
    answer = item["long_answer"]
    pubmedqa_test.append((question, context, answer))

medmcqa_test = []
for item in medmcqa:
    question = item["question"]
    options = [item["opa"], item["opb"], item["opc"], item["opd"]]
    answer_index = int(item["cop"])  
    answer = options[answer_index]
    medmcqa_test.append((question, options, answer))


In [ ]:
print("Raw PubMedQA Sample:")
for k, v in pubmedqa[0].items():
    print(f"{k}: {str(v)[:200]}")  

print("\nRaw MedMCQA Sample:")
for k, v in medmcqa[0].items():
    print(f"{k}: {v}")

Raw PubMedQA Sample:
pubid: 21645374
question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
context: {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves 
long_answer: Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of 
final_decision: yes

Raw MedMCQA Sample:
id: e9ad821a-c438-4965-9f77-760819dfa155
question: Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma
opa: Hyperplasia
opb: Hyperophy
opc: Atrophy
opd: Dyplasia
cop: 2
choice_type: single
exp: Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or func

In [5]:
def generate_answer(model, tokenizer, question, context=None, max_length=128):
    device = next(model.parameters()).device
    if context:
        input_text = f"question: {question} context: {context}"
    else:
        input_text = f"question: {question}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding="max_length", max_length=512).to(device)
    outputs = model.generate(**inputs, max_length=max_length)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.strip()
pubmedqa_preds = []
for question, context, _ in pubmedqa_test:
    pred = generate_answer(model_pubmedqa, tokenizer_pubmedqa, question, context)
    pubmedqa_preds.append(pred)


In [ ]:
def generate_mcq_answer(model, tokenizer, question, options, max_length=32):
    device = next(model.parameters()).device
    input_text = f"question: {question} options: " + " ".join([f"{chr(65+i)}. {opt}" for i, opt in enumerate(options)])
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding="max_length", max_length=512).to(device)
    outputs = model.generate(**inputs, max_length=max_length)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


    answer = answer.lower()
    for opt in options:
        if opt.lower() in answer or answer in opt.lower():
            return opt

    return answer

medmcqa_preds = []
for question, options, _ in medmcqa_test:
    pred = generate_mcq_answer(model_medmcqa, tokenizer_medmcqa, question, options)
    medmcqa_preds.append(pred)


In [ ]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")


references_pubmedqa = [ans for _, _, ans in pubmedqa_test]
predictions_pubmedqa = pubmedqa_preds

rouge_results = rouge.compute(predictions=predictions_pubmedqa, references=references_pubmedqa)
bleu_results = bleu.compute(predictions=predictions_pubmedqa, references=references_pubmedqa)


print("PubMedQA ROUGE Scores:")
print(f"ROUGE-1: {rouge_results['rouge1'] * 100:.2f}%")
print(f"ROUGE-2: {rouge_results['rouge2'] * 100:.2f}%")
print(f"ROUGE-L: {rouge_results['rougeL'] * 100:.2f}%")


print(f"PubMedQA BLEU Score: {bleu_results['bleu'] * 100:.2f}%")


correct = 0
for pred, (_, _, ans) in zip(medmcqa_preds, medmcqa_test):
    if pred.lower() == ans.lower():
        correct += 1

accuracy = correct / len(medmcqa_preds)
print(f"MedMCQA Accuracy: {accuracy * 100:.2f}%")

PubMedQA ROUGE Scores:
ROUGE-1: 26.27%
ROUGE-2: 9.49%
ROUGE-L: 20.41%
PubMedQA BLEU Score: 2.25%
MedMCQA Accuracy: 67.00%


In [ ]:
import evaluate
import numpy as np

bertscore = evaluate.load("bertscore")
f1_metric = evaluate.load("f1")

references_pubmedqa = [ans for _, _, ans in pubmedqa_test]
predictions_pubmedqa = pubmedqa_preds

bertscore_results = bertscore.compute(predictions=predictions_pubmedqa, references=references_pubmedqa, lang="en")
print("PubMedQA BERTScore:")
print(f"  Precision: {np.mean(bertscore_results['precision']):.4f}")
print(f"  Recall:    {np.mean(bertscore_results['recall']):.4f}")
print(f"  F1:        {np.mean(bertscore_results['f1']):.4f}")

pred_tokens = [pred.split() for pred in predictions_pubmedqa]
ref_tokens = [ref.split() for ref in references_pubmedqa]

def compute_f1(pred, ref):
    common = set(pred) & set(ref)
    if len(common) == 0:
        return 0.0
    precision = len(common) / len(pred)
    recall = len(common) / len(ref)
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

f1_scores = [compute_f1(p, r) for p, r in zip(pred_tokens, ref_tokens)]
avg_f1 = np.mean(f1_scores)
print(f"PubMedQA token-level F1-score: {avg_f1:.4f}")

def normalize_text(s):
    import re
    s = s.lower()
    s = re.sub(r'\s+', ' ', s) 
    s = re.sub(r'[^\w\s]', '', s) 
    return s.strip()

exact_matches = [
    normalize_text(p) == normalize_text(r)
    for p, r in zip(predictions_pubmedqa, references_pubmedqa)
]
em_score = np.mean(exact_matches)
print(f"PubMedQA Exact Match (EM) score: {em_score:.4f}")

correct = 0
for pred, (_, _, ans) in zip(medmcqa_preds, medmcqa_test):
    if pred.lower() == ans.lower():
        correct += 1
accuracy = correct / len(medmcqa_preds)
print(f"MedMCQA Accuracy: {accuracy:.2%}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PubMedQA BERTScore:
  Precision: 0.8970
  Recall:    0.8614
  F1:        0.8787
PubMedQA token-level F1-score: 0.2037
PubMedQA Exact Match (EM) score: 0.0000
MedMCQA Accuracy: 67.00%


In [ ]:
import tensorflow_hub as hub
import numpy as np
import tensorflow as tf

use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

references_pubmedqa = [ans for _, _, ans in pubmedqa_test]
predictions_pubmedqa = pubmedqa_preds

ref_embeddings = use_model(references_pubmedqa)
pred_embeddings = use_model(predictions_pubmedqa)

cosine_similarities = tf.reduce_sum(tf.multiply(ref_embeddings, pred_embeddings), axis=1) / (
    tf.norm(ref_embeddings, axis=1) * tf.norm(pred_embeddings, axis=1)
)

use_score = tf.reduce_mean(cosine_similarities).numpy()
print(f"PubMedQA USEScore (Cosine Similarity): {use_score:.4f}")


PubMedQA USEScore (Cosine Similarity): 0.4304


In [11]:
print("Sample PubMedQA predictions:")
for i in range(7):
    print(f"Q: {pubmedqa_test[i][0]}")
    print(f"GT: {pubmedqa_test[i][2]}")
    print(f"Pred: {pubmedqa_preds[i]}")
    print("-" * 60)

print("Sample MedMCQA predictions:")
for i in range(7):
    print(f"Q: {medmcqa_test[i][0]}")
    print(f"Options: {medmcqa_test[i][1]}")
    print(f"GT: {medmcqa_test[i][2]}")
    print(f"Pred: {medmcqa_preds[i]}")
    print("-" * 60)


Sample PubMedQA predictions:
Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
GT: Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the first time, we have shown the feasibility for the use of CsA in a whole plant system. Overall, our findings implicate the mitochondria as playing a critical and early role in developmentally regulated PCD in the lace plant.
Pred: Nicotine is a key component of the lace plant leaves during PCD.
------------------------------------------------------------
Q: Landolt C and snellen e acuity: differences in strabismus amblyopia?
GT: Using the charts described, there was only a sl

In [12]:
pubmedqa_test_inputs = [
    ("Does vitamin D supplementation reduce the risk of respiratory infections?", "Vitamin D is known for immune support and may affect respiratory health."),
    ("What are the long-term effects of chemotherapy on cognitive function?", "Chemotherapy is a common cancer treatment that may have side effects."),
    ("Is there a link between gut microbiota and autoimmune diseases?", "Gut microbiota influences many aspects of human health and immune function."),
    ("How effective is acupuncture in managing chronic pain?", "Acupuncture is a traditional therapy used for various pain conditions."),
    ("Can regular exercise improve insulin sensitivity in type 2 diabetes patients?", "Exercise is a recommended intervention for diabetes management.")
]

print("PubMedQA Predictions:\n")
for question, context in pubmedqa_test_inputs:
    input_text = f"question: {question} context: {context}"
    inputs = tokenizer_pubmedqa(input_text, return_tensors="pt").to(device)
    
    outputs = model_pubmedqa.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )
    pred_answer = tokenizer_pubmedqa.decode(outputs[0], skip_special_tokens=True)
    
    print(f"Q: {question}")
    print(f"Context: {context}")
    print(f"Predicted Answer: {pred_answer}")
    print("-" * 80)

PubMedQA Predictions:

Q: Does vitamin D supplementation reduce the risk of respiratory infections?
Context: Vitamin D is known for immune support and may affect respiratory health.
Predicted Answer: Vitamin D is an important component of a healthy immune system and may be a factor in the prevention of respiratory infections.
--------------------------------------------------------------------------------
Q: What are the long-term effects of chemotherapy on cognitive function?
Context: Chemotherapy is a common cancer treatment that may have side effects.
Predicted Answer: Alzheimer's disease is a common side effect of chemotherapy.
--------------------------------------------------------------------------------
Q: Is there a link between gut microbiota and autoimmune diseases?
Context: Gut microbiota influences many aspects of human health and immune function.
Predicted Answer: Gut microbiota plays an important role in the development of autoimmune diseases.
---------------------------

In [ ]:
model_medmcqa = T5ForConditionalGeneration.from_pretrained("./flan-t5-medmcqa1")
tokenizer_medmcqa = AutoTokenizer.from_pretrained("./flan-t5-medmcqa1")

model_medmcqa.to(device)

medmcqa_test_inputs = [
    ("Which vitamin deficiency causes scurvy?", ['Vitamin A', 'Vitamin C', 'Vitamin D', 'Vitamin B12']),
    ("The primary function of the sinoatrial node in the heart is to:", ['Pump blood to the lungs', 'Initiate the heartbeat', 'Regulate blood pressure', 'Filter blood']),
    ("Which antibiotic is effective against Gram-positive bacteria?", ['Ciprofloxacin', 'Penicillin', 'Metronidazole', 'Amphotericin B']),
    ("What is the most common cause of hyperthyroidism?", ['Hashimoto’s thyroiditis', 'Graves’ disease', 'Thyroid cancer', 'Iodine deficiency']),
    ("Which hormone regulates calcium levels in the blood?", ['Insulin', 'Parathyroid hormone', 'Cortisol', 'Thyroxine'])
]

print("\nMedMCQA Predictions:\n")
for question, options in medmcqa_test_inputs:
    options_text = " ".join([f"{chr(65+i)}. {opt}" for i, opt in enumerate(options)])
    input_text = f"question: {question} options: {options_text}"
    
    inputs = tokenizer_medmcqa(input_text, return_tensors="pt").to(device)
    
    outputs = model_medmcqa.generate(
        **inputs,
        max_length=50,
        num_beams=4,
        early_stopping=True
    )
    pred_answer = tokenizer_medmcqa.decode(outputs[0], skip_special_tokens=True)

    print(f"Q: {question}")
    print(f"Options: {options}")
    print(f"Predicted Answer: {pred_answer}")
    print("-" * 80)



MedMCQA Predictions:

Q: Which vitamin deficiency causes scurvy?
Options: ['Vitamin A', 'Vitamin C', 'Vitamin D', 'Vitamin B12']
Predicted Answer: Vitamin A
--------------------------------------------------------------------------------
Q: The primary function of the sinoatrial node in the heart is to:
Options: ['Pump blood to the lungs', 'Initiate the heartbeat', 'Regulate blood pressure', 'Filter blood']
Predicted Answer: Regulate blood pressure
--------------------------------------------------------------------------------
Q: Which antibiotic is effective against Gram-positive bacteria?
Options: ['Ciprofloxacin', 'Penicillin', 'Metronidazole', 'Amphotericin B']
Predicted Answer: Amphotericin B
--------------------------------------------------------------------------------
Q: What is the most common cause of hyperthyroidism?
Options: ['Hashimoto’s thyroiditis', 'Graves’ disease', 'Thyroid cancer', 'Iodine deficiency']
Predicted Answer: Graves’ disease
----------------------------

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import evaluate
import numpy as np
import tensorflow_hub as hub
import tensorflow as tf
import re

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

base_pubmedqa_preds = []
references_pubmedqa = [ans for _, _, ans in pubmedqa_test]

for context, question, _ in pubmedqa_test:
    input_text = f"question: {question} context: {context}"
    input_ids = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).input_ids
    outputs = model.generate(input_ids, max_length=64)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    base_pubmedqa_preds.append(pred)

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

rouge_results = rouge.compute(predictions=base_pubmedqa_preds, references=references_pubmedqa)
bleu_results = bleu.compute(predictions=base_pubmedqa_preds, references=references_pubmedqa)

print("PubMedQA Base Model - ROUGE Scores:")
print(f"ROUGE-1: {rouge_results['rouge1'] * 100:.2f}%")
print(f"ROUGE-2: {rouge_results['rouge2'] * 100:.2f}%")
print(f"ROUGE-L: {rouge_results['rougeL'] * 100:.2f}%")

print(f"PubMedQA Base Model - BLEU Score: {bleu_results['bleu'] * 100:.2f}%")

def compute_f1(pred, ref):
    pred_tokens = pred.split()
    ref_tokens = ref.split()
    common = set(pred_tokens) & set(ref_tokens)
    if len(common) == 0:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

f1_scores = [compute_f1(p, r) for p, r in zip(base_pubmedqa_preds, references_pubmedqa)]
avg_f1 = np.mean(f1_scores)
print(f"PubMedQA Base Model - Token-level F1-score: {avg_f1:.4f}")

def normalize_text(s):
    s = s.lower()
    s = re.sub(r'\s+', ' ', s) 
    s = re.sub(r'[^\w\s]', '', s) 
    return s.strip()

exact_matches = [
    normalize_text(p) == normalize_text(r)
    for p, r in zip(base_pubmedqa_preds, references_pubmedqa)
]
em_score = np.mean(exact_matches)
print(f"PubMedQA Base Model - Exact Match (EM) Score: {em_score:.4f}")

bertscore = evaluate.load("bertscore")
bertscore_results = bertscore.compute(predictions=base_pubmedqa_preds, references=references_pubmedqa, lang="en")
print("PubMedQA Base Model - BERTScore:")
print(f"  Precision: {np.mean(bertscore_results['precision']):.4f}")
print(f"  Recall:    {np.mean(bertscore_results['recall']):.4f}")
print(f"  F1:        {np.mean(bertscore_results['f1']):.4f}")

use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

ref_embeddings = use_model(references_pubmedqa)
pred_embeddings = use_model(base_pubmedqa_preds)

cosine_similarities = tf.reduce_sum(tf.multiply(ref_embeddings, pred_embeddings), axis=1) / (
    tf.norm(ref_embeddings, axis=1) * tf.norm(pred_embeddings, axis=1)
)

use_score = tf.reduce_mean(cosine_similarities).numpy()
print(f"PubMedQA Base Model - USEScore (Cosine Similarity): {use_score:.4f}")


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


PubMedQA Base Model - ROUGE Scores:
ROUGE-1: 17.53%
ROUGE-2: 6.45%
ROUGE-L: 14.37%
PubMedQA Base Model - BLEU Score: 0.44%
PubMedQA Base Model - Token-level F1-score: 0.1380
PubMedQA Base Model - Exact Match (EM) Score: 0.0000


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PubMedQA Base Model - BERTScore:
  Precision: 0.8643
  Recall:    0.8291
  F1:        0.8462
PubMedQA Base Model - USEScore (Cosine Similarity): 0.3191


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import evaluate
import numpy as np
import tensorflow_hub as hub
import tensorflow as tf
import re

model_path = "./flan-t5-pubmedqa1"  
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

references = [ans for _, _, ans in pubmedqa_test] 
predictions = []

for context, question, _ in pubmedqa_test:
    input_text = f"question: {question} context: {context}"
    input_ids = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).input_ids
    outputs = model.generate(input_ids, max_length=64)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions.append(pred)

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

rouge_results = rouge.compute(predictions=predictions, references=references)
bleu_results = bleu.compute(predictions=predictions, references=references)

print("Fine-tuned Model - ROUGE Scores:")
print(f"ROUGE-1: {rouge_results['rouge1'] * 100:.2f}%")
print(f"ROUGE-2: {rouge_results['rouge2'] * 100:.2f}%")
print(f"ROUGE-L: {rouge_results['rougeL'] * 100:.2f}%")

print(f"Fine-tuned Model - BLEU Score: {bleu_results['bleu'] * 100:.2f}%")

def compute_f1(pred, ref):
    pred_tokens = pred.split()
    ref_tokens = ref.split()
    common = set(pred_tokens) & set(ref_tokens)
    if len(common) == 0:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

f1_scores = [compute_f1(p, r) for p, r in zip(predictions, references)]
avg_f1 = np.mean(f1_scores)
print(f"Fine-tuned Model - Token-level F1-score: {avg_f1:.4f}")

def normalize_text(s):
    s = s.lower()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[^\w\s]', '', s)
    return s.strip()

exact_matches = [
    normalize_text(p) == normalize_text(r)
    for p, r in zip(predictions, references)
]
em_score = np.mean(exact_matches)
print(f"Fine-tuned Model - Exact Match (EM) Score: {em_score:.4f}")

bertscore = evaluate.load("bertscore")
bertscore_results = bertscore.compute(predictions=predictions, references=references, lang="en")
print("Fine-tuned Model - BERTScore:")
print(f"  Precision: {np.mean(bertscore_results['precision']):.4f}")
print(f"  Recall:    {np.mean(bertscore_results['recall']):.4f}")
print(f"  F1:        {np.mean(bertscore_results['f1']):.4f}")

use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

ref_embeddings = use_model(references)
pred_embeddings = use_model(predictions)

cosine_similarities = tf.reduce_sum(tf.multiply(ref_embeddings, pred_embeddings), axis=1) / (
    tf.norm(ref_embeddings, axis=1) * tf.norm(pred_embeddings, axis=1)
)

use_score = tf.reduce_mean(cosine_similarities).numpy()
print(f"Fine-tuned Model - USEScore (Cosine Similarity): {use_score:.4f}")


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Fine-tuned Model - ROUGE Scores:
ROUGE-1: 26.21%
ROUGE-2: 9.43%
ROUGE-L: 20.21%
Fine-tuned Model - BLEU Score: 2.32%
Fine-tuned Model - Token-level F1-score: 0.2048
Fine-tuned Model - Exact Match (EM) Score: 0.0000


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fine-tuned Model - BERTScore:
  Precision: 0.8959
  Recall:    0.8617
  F1:        0.8783
Fine-tuned Model - USEScore (Cosine Similarity): 0.4421


In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration
import torch
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

base_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base").to(device)
finetuned_model = T5ForConditionalGeneration.from_pretrained("./flan-t5-medmcqa1").to(device)


In [ ]:
def compute_accuracy(model, dataset, tokenizer):
    model.eval()
    correct = 0
    total = len(dataset)

    for sample in tqdm(dataset):
        question = sample['question']
        options = sample['options'] 
        correct_answer = sample['answer'] 

        input_text = question + " Options: " + " ".join(options)
        input_ids = tokenizer(input_text, return_tensors="pt", truncation=True).input_ids.to(device)

        with torch.no_grad():
            outputs = model.generate(input_ids, max_length=10)
        
        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

        predicted_answer = prediction.strip().upper()[0]

        if predicted_answer == correct_answer.strip().upper():
            correct += 1

    return correct / total


In [ ]:
from datasets import load_dataset

dataset = load_dataset("medmcqa", split="validation[:100]")

label_map = {0: "A", 1: "B", 2: "C", 3: "D"}

test_dataset = []
for row in dataset:
    options = [
        f"A. {row['opa']}",
        f"B. {row['opb']}",
        f"C. {row['opc']}",
        f"D. {row['opd']}"
    ]
    test_dataset.append({
        "question": row["question"],
        "options": options,
        "answer": label_map[row["cop"]]  
    })


In [ ]:
print("Evaluating Base FLAN-T5...")
base_acc = compute_accuracy(base_model, test_dataset, tokenizer)
print(f"Base Model Accuracy on MedMCQA: {base_acc * 100:.2f}%")


Evaluating Base FLAN-T5...


100%|██████████| 100/100 [00:06<00:00, 15.75it/s]

Base Model Accuracy on MedMCQA: 24.00%
